In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================

# --- Dataset sampling ---
N_SAMPLES = 25000          # how many single-sentence clips to use, 25000 = 9%
MIN_UP_VOTES = 2           # quality filter
MAX_DOWN_VOTES = 0
MIN_SENTENCE_LEN = 10
MAX_SENTENCE_LEN = 200

# --- Multi-sentence concatenation (fixes the mid-sentence cutoff bug) ---
CONCAT_FRACTION = 0.3      # 30% of N_SAMPLES will instead be built as 2-3 sentence combos
CONCAT_MIN_CLIPS = 2
CONCAT_MAX_CLIPS = 3

# --- Training ---
# Point at the base model for a fresh run, or at your last checkpoint to continue improving it
BASE_MODEL_PATH = "./models/MOSS-TTS-Nano"
# BASE_MODEL_PATH = "./output/moss_tts_nano_sft/checkpoint-last"  # <- use this to continue from round 1

CODEC_PATH = "./models/MOSS-Audio-Tokenizer-Nano"
OUTPUT_DIR = "./output/moss_tts_nano_sft_v2"
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_LENGTH = 1024
NUM_PROCESSES = 2           # both T4s — safe now since we're in fp32, no grad-scaler conflict
MIXED_PRECISION = "no"      # T4 doesn't support bf16; fp16 broke grad scaling; fp32 is what worked

# --- Paths ---
KAGGLE_DATASET = "amirftma/common-voice-fa-v13"
WORKDIR = "MOSS-TTS-Nano"

In [ ]:
# ============================================================
# CELL 2 — Imports & installs 
# ============================================================
import os
import sys
import json
import random
import subprocess

import pandas as pd
import IPython.display as ipd

def run(cmd, cwd=None):
    """Stream subprocess output live instead of buffering it silently."""
    print(f"$ {cmd}")
    return os.system(f"cd {cwd} && {cmd}" if cwd else cmd)

# Core deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "accelerate", "datasets", "soundfile", "kagglehub"], check=True)

# Clone the repo (skip if already present from a prior session)
if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "https://github.com/OpenMOSS/MOSS-TTS-Nano.git", WORKDIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{WORKDIR}/requirements.txt"], check=True)
    # Fix the torch/torchvision version mismatch discovered earlier
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision==0.22.0"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())